In [1]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import random
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
import matplotlib.colors as mcolors
from sklearn.metrics import accuracy_score

In [2]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True

In [3]:

X, y = make_blobs(n_samples=2000, centers=[[-2, 2], [-6, 6], [5.5, 4], [-4, -4], [5, -1.0]], cluster_std=[1.5, 1.0, 1.5, 1.5, 1.5], random_state=seed)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train, dtype=torch.long).to(device)
X_test = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test = torch.tensor(y_test, dtype=torch.long).to(device)


In [4]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [5]:

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MLP, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
        
    def forward(self, x):
        return self.layers(x)


def calculate_accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        outputs = model(X)
        _, predicted = torch.max(outputs, 1)
    accuracy = accuracy_score(y.cpu(), predicted.cpu())
    class_accuracies = []
    for class_idx in range(5):
        mask = (y == class_idx)
        class_accuracy = accuracy_score(y[mask].cpu(), predicted[mask].cpu())
        class_accuracies.append(class_accuracy)
    return class_accuracies


def plot_decision_boundary(model, X, y, class_names, title, unlearning_class=None, unlearned_predictions=None):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.01),
                         np.arange(y_min, y_max, 0.01))
    grid = np.c_[xx.ravel(), yy.ravel()]
    

    grid_tensor = torch.tensor(grid, dtype=torch.float32).to(device)
    with torch.no_grad():
        Z = model(grid_tensor)
        _, Z = torch.max(Z, 1)
        Z = Z.cpu().numpy().reshape(xx.shape)  

    colors = plt.get_cmap("tab10").colors
    cmap = mcolors.ListedColormap(colors[:len(class_names)])


    bounds = np.arange(len(class_names) + 1) - 0.5
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    plt.figure(figsize=(6, 3))
    plt.contourf(xx, yy, Z, levels=bounds, cmap=cmap, norm=norm, alpha=0.2)  
    plt.contour(xx, yy, Z, colors='k', linewidths=0.8)  #

    legend_handles = []
    for i, class_name in enumerate(class_names):
        mask = (y == i)
        if unlearning_class is not None and unlearning_class == i:

            handle = plt.scatter(X[mask, 0], X[mask, 1], color=colors[i], edgecolors=None, marker='*', s=80,  label=class_name)
        else:
            handle = plt.scatter(X[mask, 0], X[mask, 1], color=colors[i], edgecolors=None, marker='o', s=40,  label=class_name)
        legend_handles.append(handle)
 

    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())


    accuracies = calculate_accuracy(model, X_train, y_train)
    

    labels_with_acc = [f'{class_names[i]} ({accuracies[i]*100:.2f}%)' for i in range(len(class_names))]
    plt.legend(legend_handles, labels_with_acc, loc='lower right', fontsize=10)


    plt.xlabel(r"$z_1$", fontsize=20)
    plt.ylabel(r"$z_2$", fontsize=20)
    plt.xticks(fontsize=15)
    plt.yticks(fontsize=15)

    plt.savefig(f'class_toy_figure/{title}.png',bbox_inches='tight')
    plt.savefig(f'class_toy_figure/{title}.pdf',bbox_inches='tight')
    plt.show()


In [ ]:

model = MLP(input_dim=2, hidden_dim=16, output_dim=5).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')



In [ ]:
unlearning_class = 2

#### CUP

In [9]:
from mode_toy.CUP import Cup

In [ ]:
cup_model = copy.deepcopy(model)
base_optimizer = optim.Adam(cup_model.parameters(), lr=0.01)
cup_optmizer = Cup(base_optimizer, alpha=0.7)
unlearning_mask = (y_train == unlearning_class)
retain_mask = ~unlearning_mask

num_unlearning_epochs = 20
for epoch in range(num_unlearning_epochs):
    base_optimizer.zero_grad()
    
    # Unlearning loss: 해당 클래스에 대한 손실을 최대화하려는 시도
    outputs_unlearning = cup_model(X_train[unlearning_mask])
    unlearning_loss = -criterion(outputs_unlearning, y_train[unlearning_mask])
    
    # Retain loss: 나머지 클래스에 대한 손실을 최소화하려는 시도
    outputs_retain = cup_model(X_train[retain_mask])
    retain_loss = criterion(outputs_retain, y_train[retain_mask])
    losses = [unlearning_loss, retain_loss]
    cup_optmizer.step(losses)
    #total_loss.backward()
    #unlearning_optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f'Unlearning Epoch [{epoch+1}/{num_unlearning_epochs}], Retain Loss: {retain_loss.item():.4f}')

#### Visualization

In [ ]:

class_names = ['Class 0', 'Class 1', 'Class 2', 'Class 3', 'Class 4']

print("Original Model Decision Boundary:")
plot_decision_boundary(model, X_train.cpu().numpy(), y_train.cpu().numpy(), class_names, "Original Model Decision Boundary",unlearning_class=unlearning_class)


In [ ]:

cup_model.eval()
with torch.no_grad():
    outputs = cup_model(X_train[unlearning_mask])
    _, cup_predictions = torch.max(outputs, 1)
cup_predictions = cup_predictions.cpu().numpy()

print("Unlearning Model Decision Boundary:")
plot_decision_boundary(cup_model, X_train.cpu().numpy(), y_train.cpu().numpy(), class_names, 
                       "cup Model Decision Boundary", 
                       unlearning_class=unlearning_class, 
                       unlearned_predictions=cup_predictions)